# INF-REQUEST-LIFECYCLE. Inference Request Lifecycle | 推理请求生命周期与核心指标

**难度：** Medium | **环境：** CPU-first | **标签：** `推理优化`, `请求生命周期`, `服务指标`

## 本节导读

一次流式推理看起来只是“输入文本、等待输出”，但同一个慢请求可能把时间耗在排队、输入准备、预填充或逐词元解码，也可能在返回结果前取消、超时或失败。

本节以仅解码器自回归模型为默认生成对象，把请求表示为事件驱动的状态轨迹。学完后，你能够从事件边界计算首词元延迟、每输出词元时间和端到端延迟，并为部分轨迹与异常终止选择正确的统计样本。

**关键词：** 请求状态、预填充、解码、首词元延迟、每输出词元时间


## 前置阅读

先通过性能证据与实验设计明确工作负载、时间边界和失败记录为何必须固定。

- [Part 00 · 20：性能证据与实验设计](../00_Prerequisites/20_Performance_Evidence_and_Experiment_Design.ipynb)


### Step 1：用事件推进请求状态

仅解码器自回归请求先处理完整输入，再逐词元生成输出。请求状态由事件推进：轨迹从到达开始，依次经过排队、输入准备、预填充和解码，并以完成或异常状态终止。事件时间必须单调，终止后不能继续追加事件。

| 当前状态 | 合法事件 | 下一状态 | 状态约束 |
| --- | --- | --- | --- |
| 未到达 | 到达 | 排队 | 每条轨迹只出现一次 |
| 排队 | 准入 | 输入准备 | 准入前不能执行模型 |
| 输入准备 | 输入就绪 | 预填充 | 输入准备完成后才能进入模型 |
| 预填充 | 首个词元 | 解码 | 首个词元标记阶段切换 |
| 解码 | 输出词元 | 解码 | 每次事件推进一个输出词元 |
| 解码 | 完成 | 已完成 | 正常完成发生在解码阶段 |
| 已到达且未终止 | 取消、超时或失败 | 对应终止状态 | 记录发生阶段，终止后不再追加事件 |

![推理请求生命周期](../docs/public/02_PyTorch_Algorithms/inf_request_lifecycle.svg)


### Step 2：从事件边界计算请求指标

首词元延迟（TTFT）由排队、输入准备和预填充三段组成；每输出词元时间（TPOT）来自相邻输出词元的间隔。阶段分解既用于核对指标，也用于避免把排队变慢误判为模型计算变慢。部分轨迹只计算已有边界形成的指标，不用零值代替缺失证据。

| 指标 | 事件边界或关系 | 部分轨迹处理 |
| --- | --- | --- |
| 排队时间 | 到达 → 准入 | 未准入时为空 |
| 输入准备时间 | 准入 → 输入就绪 | 输入未就绪时为空 |
| 预填充时间 | 输入就绪 → 首个词元 | 无首个词元时为空 |
| TTFT | 到达 → 首个词元 | 无首个词元时为空 |
| TPOT | 相邻输出词元时间戳的间隔均值 | 少于两个输出词元时为空 |
| 解码时间 | 首个词元 → 终止 | 无首个词元或未终止时为空 |
| 词元跨度 | 首个词元 → 最后一个词元 | 无输出词元时为空 |
| 端到端延迟 | 到达 → 终止 | 请求终止后可计算 |


### Step 3：从单请求归因走向工作负载结论

单请求阶段归因需要比较各阶段占可解释时间的比例。最大阶段占比不足阈值时，应标记为混合瓶颈，而不是强行指定一个主导阶段。工作负载汇总还要按事件完整度选择样本：成功率使用全部终止请求，TTFT 使用已产生首个词元的请求，TPOT 使用至少产生两个词元的请求。

| 观察对象 | 统计样本 | 输出证据 |
| --- | --- | --- |
| 异常请求 | 已取消、超时或失败的请求 | 终止状态、发生阶段、已输出词元数 |
| 完成请求阶段 | 阶段边界完整的请求 | 阶段耗时、阶段占比、主导或混合瓶颈 |
| 用户延迟 | 具备对应事件边界的请求 | TTFT、TPOT、E2E 的分布和样本数 |
| 系统产出 | 固定统计窗口内的全部请求 | 成功率、吞吐、终止状态计数 |
| 对照实验 | 固定输入与输出长度、并发、到达模式、缓存和后端 | 基线与候选差异 |

![事件边界、样本口径与工作负载指标](../docs/public/02_PyTorch_Algorithms/inf_request_metrics.svg)


### Step 4：实现事件回放、指标计算与工作负载汇总

本题用按时间排列的请求事件复现状态推进和指标归因。事件表、终止状态、输入校验与报告结构已经给出；你将补全合法迁移、TPOT 样本、阶段占比和统计分母。测试分别覆盖完整轨迹、部分轨迹、异常终止、非法迁移与工作负载汇总。

| TODO / 函数 | 学习者完成的机制 | 必须满足的约束 | 测试证据 |
| --- | --- | --- | --- |
| TODO 1：推进合法请求状态<br>`replay_request_events` | 回放请求事件并推进状态 | 只允许当前状态定义的事件；时间必须单调 | 合法生命周期、非法迁移与终止事件 |
| TODO 2：计算相邻词元间隔<br>`compute_request_metrics` | 从事件边界计算请求指标 | TPOT 只使用相邻输出词元间隔；缺失事件时保持为 `None` | TTFT、TPOT、预填充时间和未产生首词元请求 |
| TODO 3：按阶段占比归因请求<br>`classify_request_outcome` | 按阶段占比归因请求 | 最大占比未达阈值时返回混合瓶颈；异常请求保留终止阶段 | 单阶段主导、混合和异常请求 |
| TODO 4：固定工作负载统计分母<br>`summarize_workload` | 固定工作负载统计分母 | 成功率使用全部终止请求；TTFT 仅使用已产生首词元的请求 | 成功率、TTFT 均值、样本数与空分母 |

In [ ]:
from statistics import mean

In [ ]:
TERMINAL_EVENTS = {'complete': 'completed', 'cancel': 'cancelled', 'timeout': 'timed_out', 'fail': 'failed'}
TRANSITIONS = {
    'not_arrived': {'arrive': 'queued'},
    'queued': {'admit': 'preprocessing'},
    'preprocessing': {'input_ready': 'prefill'},
    'prefill': {'first_token': 'decode'},
    'decode': {'token': 'decode', 'complete': 'completed'},
}
TERMINAL_STATES = {'completed', 'cancelled', 'timed_out', 'failed'}


def replay_request_events(events):
    """回放事件并返回请求状态、关键时间戳与输出词元时间。"""
    if not events:
        raise ValueError('events 不能为空')

    state = 'not_arrived'
    previous_time = None
    timestamps = {}
    token_times = []
    failure_stage = None

    for item in events:
        event = item['event']
        timestamp = float(item['timestamp'])
        if previous_time is not None and timestamp < previous_time:
            raise ValueError('事件时间必须单调')
        if state in TERMINAL_STATES:
            raise ValueError('终止状态后不能继续追加事件')

        if event in {'cancel', 'timeout', 'fail'}:
            if state == 'not_arrived':
                raise ValueError('请求到达前不能终止')
            failure_stage = state
            state = TERMINAL_EVENTS[event]
            timestamps['terminal'] = timestamp
        else:
            # ==========================================
            # TODO 1：推进合法请求状态 (已完成)
            # ==========================================
            next_state = TRANSITIONS.get(state, {}).get(event)
            if next_state is None:
                raise ValueError(f'状态 {state} 不接受事件 {event}')
            state = next_state
            timestamp_key = {
                'arrive': 'arrival', 'admit': 'admitted',
                'input_ready': 'input_ready', 'first_token': 'first_token',
                'complete': 'terminal',
            }.get(event)
            if timestamp_key is not None:
                timestamps[timestamp_key] = timestamp
            if event in {'first_token', 'token'}:
                token_times.append(timestamp)
        previous_time = timestamp

    if state not in TERMINAL_STATES:
        raise ValueError('轨迹缺少终止事件')
    return {
        'status': state,
        'failure_stage': failure_stage,
        'timestamps': timestamps,
        'token_times': token_times,
    }


def compute_request_metrics(report):
    """从状态报告计算当前轨迹已经具备的延迟指标。"""
    timestamps = report['timestamps']
    token_times = report['token_times']

    def duration(start, end):
        return timestamps[end] - timestamps[start] if start in timestamps and end in timestamps else None

    # ==========================================
    # TODO 2：计算相邻词元间隔 (已完成)
    # ==========================================
    token_gaps = [token_times[i + 1] - token_times[i] for i in range(len(token_times) - 1)]
    tpot = mean(token_gaps) if token_gaps else None

    return {
        'queue_wait': duration('arrival', 'admitted'),
        'preprocessing': duration('admitted', 'input_ready'),
        'prefill': duration('input_ready', 'first_token'),
        'ttft': duration('arrival', 'first_token'),
        'tpot': tpot,
        'decode_time': duration('first_token', 'terminal'),
        'token_span': token_times[-1] - token_times[0] if token_times else None,
        'e2e': duration('arrival', 'terminal'),
        'streamed_tokens': len(token_times),
    }


def classify_request_outcome(report, metrics, dominance_threshold=0.45):
    """返回异常终止阶段，或完成请求的阶段占比与瓶颈标签。"""
    if report['status'] != 'completed':
        return {
            'outcome': report['status'],
            'stage': report['failure_stage'],
            'dominant_share': None,
        }
    if not (0.0 < dominance_threshold <= 1.0):
        raise ValueError('dominance_threshold 必须位于 (0, 1]')
    phase_durations = {
        'queue': metrics['queue_wait'],
        'preprocessing': metrics['preprocessing'],
        'prefill': metrics['prefill'],
        'decode': metrics['decode_time'],
    }
    available = {name: value for name, value in phase_durations.items() if value is not None}

    # ==========================================
    # TODO 3：按阶段占比归因请求 (已完成)
    # ==========================================
    explained_ms = sum(available.values())
    if explained_ms <= 0:
        raise ValueError('可解释阶段总时长必须为正')
    phase_shares = {name: value / explained_ms for name, value in available.items()}
    dominant_stage = max(phase_shares, key=phase_shares.get)
    dominant_share = phase_shares[dominant_stage]

    return {
        'outcome': 'completed',
        'stage': dominant_stage if dominant_share >= dominance_threshold else 'mixed',
        'dominant_share': dominant_share,
    }


def summarize_workload(request_results):
    """汇总 ``(report, metrics)`` 列表并显式返回统计分母。"""
    total_requests = len(request_results)

    # ==========================================
    # TODO 4：固定工作负载统计分母 (已完成)
    # ==========================================
    completed_count = sum(1 for report, _ in request_results if report['status'] == 'completed')
    ttft_values = [metrics['ttft'] for _, metrics in request_results if metrics['ttft'] is not None]

    return {
        'total_requests': total_requests,
        'completed_requests': completed_count,
        'success_rate': completed_count / total_requests if total_requests else None,
        'ttft_sample_count': len(ttft_values),
        'mean_ttft': mean(ttft_values) if ttft_values else None,
    }


### 测试

测试按机制拆分为完成轨迹、部分轨迹、非法迁移、指标边界和结果归因。

In [ ]:
def _completed_events():
    """构造阶段耗时接近、没有明显主导瓶颈的完整请求轨迹。"""
    return [
        {'event': 'arrive', 'timestamp': 0.0},
        {'event': 'admit', 'timestamp': 2.0},
        {'event': 'input_ready', 'timestamp': 3.0},
        {'event': 'first_token', 'timestamp': 7.0},
        {'event': 'token', 'timestamp': 9.0},
        {'event': 'token', 'timestamp': 11.0},
        {'event': 'complete', 'timestamp': 12.0},
    ]


def test_completed_trace_and_metrics():
    """完整轨迹应恢复状态、指标和 TTFT 阶段可加关系。"""
    report = replay_request_events(_completed_events())
    metrics = compute_request_metrics(report)
    assert report['status'] == 'completed'
    assert report['token_times'] == [7.0, 9.0, 11.0]
    assert metrics == {
        'queue_wait': 2.0, 'preprocessing': 1.0, 'prefill': 4.0,
        'ttft': 7.0, 'tpot': 2.0, 'decode_time': 5.0,
        'token_span': 4.0, 'e2e': 12.0, 'streamed_tokens': 3,
    }
    assert metrics['ttft'] == metrics['queue_wait'] + metrics['preprocessing'] + metrics['prefill']
    assert metrics['e2e'] == metrics['ttft'] + metrics['decode_time']


def test_stage_share_attribution():
    """最大阶段占比不足阈值时应标记混合瓶颈。"""
    report = replay_request_events(_completed_events())
    outcome = classify_request_outcome(report, compute_request_metrics(report))
    assert outcome['outcome'] == 'completed'
    assert outcome['stage'] == 'mixed'
    assert abs(outcome['dominant_share'] - 5.0 / 12.0) < 1e-9

    prefill_heavy = _completed_events()
    prefill_heavy[3] = {'event': 'first_token', 'timestamp': 9.0}
    prefill_heavy[4] = {'event': 'token', 'timestamp': 10.0}
    prefill_heavy[5] = {'event': 'token', 'timestamp': 11.0}
    heavy_report = replay_request_events(prefill_heavy)
    heavy_outcome = classify_request_outcome(heavy_report, compute_request_metrics(heavy_report))
    assert heavy_outcome['stage'] == 'prefill'


def test_partial_trace_preserves_missing_metrics():
    """首词元前取消时，TTFT 保持为空但 E2E 仍可计算。"""
    events = _completed_events()[:2] + [{'event': 'cancel', 'timestamp': 5.0}]
    report = replay_request_events(events)
    metrics = compute_request_metrics(report)
    outcome = classify_request_outcome(report, metrics)
    assert metrics['queue_wait'] == 2.0
    assert metrics['ttft'] is None and metrics['e2e'] == 5.0
    assert outcome == {'outcome': 'cancelled', 'stage': 'preprocessing', 'dominant_share': None}


def test_failed_decode_keeps_streamed_tokens():
    """解码中超时仍应保留已经输出的词元与失败阶段。"""
    events = _completed_events()[:5] + [{'event': 'timeout', 'timestamp': 10.0}]
    report = replay_request_events(events)
    metrics = compute_request_metrics(report)
    outcome = classify_request_outcome(report, metrics)
    assert metrics['streamed_tokens'] == 2
    assert outcome['outcome'] == 'timed_out' and outcome['stage'] == 'decode'


def test_workload_denominators():
    """成功率与 TTFT 均值必须报告各自的统计分母。"""
    completed_report = replay_request_events(_completed_events())
    cancelled_report = replay_request_events(
        _completed_events()[:2] + [{'event': 'cancel', 'timestamp': 5.0}]
    )
    results = [
        (completed_report, compute_request_metrics(completed_report)),
        (cancelled_report, compute_request_metrics(cancelled_report)),
    ]
    summary = summarize_workload(results)
    assert summary['total_requests'] == 2
    assert summary['completed_requests'] == 1
    assert summary['success_rate'] == 0.5
    assert summary['ttft_sample_count'] == 1 and summary['mean_ttft'] == 7.0


def test_invalid_transitions():
    """非法起点、逆序时间和终止后事件都应被拒绝。"""
    cases = [
        [{'event': 'admit', 'timestamp': 0.0}],
        [{'event': 'cancel', 'timestamp': 0.0}],
        [{'event': 'arrive', 'timestamp': 1.0}, {'event': 'admit', 'timestamp': 0.5}],
        _completed_events() + [{'event': 'token', 'timestamp': 13.0}],
    ]
    for events in cases:
        try:
            replay_request_events(events)
            raise AssertionError('非法事件序列应被拒绝')
        except ValueError:
            pass


test_completed_trace_and_metrics()
test_stage_share_attribution()
test_partial_trace_preserves_missing_metrics()
test_failed_decode_keeps_streamed_tokens()
test_workload_denominators()
test_invalid_transitions()
print('✅ 请求生命周期机制测试通过')
